<a href="https://colab.research.google.com/github/pranusha751/ITA0601-machine-learning/blob/main/EX_2(ML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
from itertools import product

# Step 1: Create and save the training dataset as a CSV file
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Cool'],
    'Forecast': ['Same', 'Same', 'Change', 'Change'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

pd.DataFrame(data).to_csv('training_data.csv', index=False)

# Step 2: Read the CSV file
df = pd.read_csv('training_data.csv')

print("TRAINING DATASET:")
print(df.to_string(index=False))

attributes = list(df.columns[:-1])
target = df.columns[-1]

# Find possible values for every attribute
domains = [
    list(df[attr].unique()) for attr in attributes
]

# A hypothesis covers an example if every attribute matches
# or the hypothesis contains '?'
def covers(h, x):
    return all(
        h[i] == '?' or h[i] == x[i]
        for i in range(len(h))
    )

# h1 is at least as general as h2
def more_general(h1, h2):
    return all(
        a == '?' or a == b
        for a, b in zip(h1, h2)
    )

# Remove hypotheses that are less general than another
def prune_G(G):
    result = []
    for g in G:
        if not any(
            g != other and more_general(other, g)
            for other in G
        ):
            result.append(g)
    return result

# Step 3: Initialize boundaries
S = [('Ø',) * len(attributes)]
G = [('?',) * len(attributes)]

# Step 4: Process each training example
for _, row in df.iterrows():
    x = tuple(row[attr] for attr in attributes)
    label = row[target].strip().lower()

    if label == 'yes':
        # Remove G hypotheses that do not cover positive example
        G = [g for g in G if covers(g, x)]

        new_S = []
        for s in S:
            if covers(s, x):
                new_S.append(s)
            else:
                h = list(s)
                for i in range(len(h)):
                    if h[i] == 'Ø':
                        h[i] = x[i]
                    elif h[i] != x[i]:
                        h[i] = '?'
                h = tuple(h)

                if any(more_general(g, h) for g in G):
                    new_S.append(h)

        S = list(set(new_S))

        # Keep only maximally specific hypotheses in S
        S = [
            s for s in S
            if not any(
                s != other and more_general(s, other)
                for other in S
            )
        ]

    else:
        # Remove S hypotheses that cover negative example
        S = [s for s in S if not covers(s, x)]

        new_G = []
        for g in G:
            if not covers(g, x):
                new_G.append(g)
            else:
                # Generate minimal specializations of g
                for i in range(len(g)):
                    if g[i] == '?':
                        for value in domains[i]:
                            if value != x[i]:
                                h = list(g)
                                h[i] = value
                                h = tuple(h)

                                if any(
                                    more_general(h, s) for s in S
                                ):
                                    new_G.append(h)

        G = prune_G(list(set(new_G)))

# Step 5: Display final boundaries
print("\nFINAL SPECIFIC BOUNDARY (S):")
for h in S:
    print(list(h))

print("\nFINAL GENERAL BOUNDARY (G):")
for h in G:
    print(list(h))

print("\nHYPOTHESES CONSISTENT WITH THE TRAINING DATA")
print("The version space is bounded by S and G.")


TRAINING DATASET:
  Sky AirTemp Humidity   Wind Water Forecast EnjoySport
Sunny    Warm   Normal Strong  Warm     Same        Yes
Sunny    Warm     High Strong  Warm     Same        Yes
Rainy    Cold     High Strong  Warm   Change         No
Sunny    Warm     High Strong  Cool   Change        Yes

FINAL SPECIFIC BOUNDARY (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

FINAL GENERAL BOUNDARY (G):
['?', 'Warm', '?', '?', '?', '?']
['Sunny', '?', '?', '?', '?', '?']

HYPOTHESES CONSISTENT WITH THE TRAINING DATA
The version space is bounded by S and G.
